In [1]:
# Install needed libraries
import pandas as pd

In [2]:
# Compile transcriptome tsvs into one pandas dataframe

# Read the files
df1 = pd.read_csv('Data/TCGA_Transcriptome/TCGA-KICH.star_fpkm-uq.tsv', sep='\t')
df2 = pd.read_csv('Data/TCGA_Transcriptome/TCGA-KIRC.star_fpkm-uq.tsv', sep='\t')
df3 = pd.read_csv('Data/TCGA_Transcriptome/TCGA-KIRP.star_fpkm-uq.tsv', sep='\t')

# Get the name of the first (gene name) column
gene_col = df1.columns[0]

# Set gene name as index for all three
df1 = df1.set_index(gene_col)
df2 = df2.set_index(gene_col)
df3 = df3.set_index(gene_col)

# Concatenate along columns (axis=1)
combined_df = pd.concat([df1, df2, df3], axis=1)

# Reset index to make gene names a column again
combined_df1 = combined_df.reset_index()


print(f"\nTranscriptome combined dataframe shape: {combined_df1.shape}")


Transcriptome combined dataframe shape: (60660, 1025)


/tmp/ipykernel_7913/1647897834.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df1 = combined_df.reset_index()


In [3]:
# Combine clinical data tsvs into one pandas dataframe and drop all unneeded columns

# Read the files
df1 = pd.read_csv('Data/TCGA_Clinical/TCGA-KICH.clinical.tsv', sep='\t')
df2 = pd.read_csv('Data/TCGA_Clinical/TCGA-KIRC.clinical.tsv', sep='\t')
df3 = pd.read_csv('Data/TCGA_Clinical/TCGA-KIRP.clinical.tsv', sep='\t')

# Assign a list of columns we will keep
columns_to_keep = ['sample', 'primary_diagnosis.diagnoses']

# Create new dataframes that ONLY have the columns we want
df1a = df1[columns_to_keep]
df2a = df2[columns_to_keep]
df3a = df3[columns_to_keep]

# Concatenate along rows (axis=0)
combined_df2 = pd.concat([df1a, df2a, df3a], axis=0)


print(f"\nClinical combined dataframe shape: {combined_df2.shape}")


Clinical combined dataframe shape: (1407, 2)


In [4]:
# Make sure only samples present in BOTH dataframes are included
# Extract names from each dataframe
names_in_columns = set(combined_df1.columns)  # names from df1's column headers
first_col = combined_df2.columns[0]           # get the name of df2's first column
names_in_rows = set(combined_df2[first_col])  # names from df2's first column

# Find names present in BOTH
common_names = names_in_columns & names_in_rows

print(f"Names in df1 (columns): {len(names_in_columns)}")
print(f"Names in df2 (rows):    {len(names_in_rows)}")
print(f"Names in common:        {len(common_names)}")

# Filter df1: keep only columns whose header is a common name
# (preserve any non-name columns like a gene/ID column if needed)
non_name_cols = [combined_df1.columns[0]]  # e.g., keep the first column (gene names, etc.)
name_cols_to_keep = [c for c in combined_df1.columns if c in common_names]
filtered_df1 = combined_df1[non_name_cols + name_cols_to_keep]

# Filter df2: keep only rows where the first column value is a common name
non_name_rows = combined_df2.iloc[[0]]  # preserve row 1
rest_filtered = combined_df2.iloc[1:]  # everything after row 1
rest_filtered = rest_filtered[rest_filtered[first_col].isin(common_names)]
filtered_df2 = pd.concat([non_name_rows, rest_filtered], ignore_index=True)\

print(f"\ndf1 filtered shape: {filtered_df1.shape}")
print(f"df2 filtered shape: {filtered_df2.shape}")


Names in df1 (columns): 1025
Names in df2 (rows):    1407
Names in common:        1024

df1 filtered shape: (60660, 1025)
df2 filtered shape: (1025, 2)


In [5]:
# MERGE dataframes

# transpose dataframe
filtered_df1_t = filtered_df1.T

# reset index 
filtered_df1_t_reset = filtered_df1_t.reset_index()

# Set column names as the first row
filtered_df1_t_reset.columns = filtered_df1_t_reset.iloc[0]

# Only take names after the first row
filtered_df1_t_reset_final = filtered_df1_t_reset.iloc[1:]

# Show shape of transposed dataset
print(f'Shape of transposed dataset: {filtered_df1_t_reset_final.shape}')

# set names for clinicla data so Ensembl_ID matches transcriptome data
filtered_df2.columns = ['Ensembl_ID', 'Cancer_Type']

# Merge data on esnembl id
merged_df = pd.merge(
    filtered_df1_t_reset_final,
    filtered_df2,
    on="Ensembl_ID",
    how="inner"   
)

# Set index as Ensembl_ID
merged_df.set_index('Ensembl_ID', inplace=True)

# Lost 1 sample because transcriptome didn't have it
# Added 1 column the label (Cancer type)
print(f'Shape after merging: {merged_df.shape}')

merged_df.to_csv('Data/merged_data.csv')

Shape of transposed dataset: (1024, 60661)
Shape after merging: (1024, 60661)
